# 📈 FDS Project: Stock Price Prediction — AI Agent
### Green Energy Sector | NSE India Stocks
**Stocks:** ADANIGREEN | INOXGREEN | JSWENERGY | SUZLON

---
| Review | Task | Status |
|--------|------|--------|
| Review 1 | Advanced Algorithms (LSTM, XGBoost, Random Forest, SVR) | ✅ |
| Review 2 | GAN-Generated Synthetic Dataset + Algorithm Comparison | ✅ |
| Review 3 | Text Mining (Sentiment Analysis on Stock News) | ✅ |
| AI Agent | Autonomous Stock Prediction & Recommendation Agent | ✅ |

## 🔧 Step 1: Install Required Libraries

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost tensorflow torch textblob nltk joblib

## 📦 Step 2: Import All Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML Algorithms
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# XGBoost
import xgboost as xgb

# Deep Learning (LSTM)
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# Text Mining
import nltk
from textblob import TextBlob
import re
import string

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import random
import torch
import torch.nn as nn

print('✅ All libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')
print(f'XGBoost version: {xgb.__version__}')

## 📂 Step 3: Load All 4 Stock CSV Files

In [ ]:
# ---- Set path to your FDS folder ----
DATA_PATH = 'FDS/'   # Adjust if needed

stock_files = {
    'ADANIGREEN': 'ADANIGREEN.NS_stock_data.csv',
    'INOXGREEN':  'INOXGREEN.NS_stock_data.csv',
    'JSWENERGY':  'JSWENERGY.NS_stock_data.csv',
    'SUZLON':     'SUZLON.NS_stock_data.csv'
}

stock_dfs = {}
for name, file in stock_files.items():
    path = os.path.join(DATA_PATH, file)
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    # Parse date column (handles multiple formats)
    for col in df.columns:
        if 'date' in col.lower():
            df[col] = pd.to_datetime(df[col], infer_datetime_format=True)
            df.rename(columns={col: 'Date'}, inplace=True)
            break
    df.sort_values('Date', inplace=True)
    df.reset_index(drop=True, inplace=True)
    stock_dfs[name] = df
    print(f'✅ {name}: {df.shape[0]} rows | Columns: {list(df.columns)}')

print('\n📊 Sample ADANIGREEN data:')
stock_dfs['ADANIGREEN'].head()

## 🔍 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

for i, (name, df) in enumerate(stock_dfs.items()):
    axes[i].plot(df['Date'], df['Close'], color=colors[i], linewidth=1.5, label='Close Price')
    axes[i].fill_between(df['Date'], df['Close'], alpha=0.1, color=colors[i])
    axes[i].set_title(f'{name} — Closing Price Over Time', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('Price (INR)')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('📈 Green Energy Stocks — Historical Price Trends', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('stock_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA Plot saved!')

In [ ]:
# Statistical Summary
for name, df in stock_dfs.items():
    print(f'\n📊 {name} — Statistics:')
    print(df[['Open','High','Low','Close','Volume']].describe().round(2))

## ⚙️ Step 5: Feature Engineering (Technical Indicators)

In [ ]:
def add_technical_indicators(df):
    df = df.copy()
    
    # Moving Averages
    df['MA_7']  = df['Close'].rolling(window=7).mean()
    df['MA_21'] = df['Close'].rolling(window=21).mean()
    df['MA_50'] = df['Close'].rolling(window=50).mean()
    
    # Exponential Moving Average
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    
    # MACD
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain  = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss  = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs    = gain / (loss + 1e-8)
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    df['BB_Mid']   = df['Close'].rolling(window=20).mean()
    df['BB_Upper'] = df['BB_Mid'] + 2 * df['Close'].rolling(window=20).std()
    df['BB_Lower'] = df['BB_Mid'] - 2 * df['Close'].rolling(window=20).std()
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']
    
    # Price Returns
    df['Daily_Return']  = df['Close'].pct_change()
    df['Weekly_Return'] = df['Close'].pct_change(5)
    
    # Volatility
    df['Volatility'] = df['Daily_Return'].rolling(window=14).std()
    
    # Price Momentum
    df['Momentum'] = df['Close'] - df['Close'].shift(10)
    
    # Volume features
    df['Volume_MA'] = df['Volume'].rolling(window=10).mean()
    df['Volume_Ratio'] = df['Volume'] / (df['Volume_MA'] + 1e-8)
    
    # Target: Next day closing price
    df['Target'] = df['Close'].shift(-1)
    
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

processed_dfs = {}
for name, df in stock_dfs.items():
    processed_dfs[name] = add_technical_indicators(df)
    print(f'✅ {name}: {processed_dfs[name].shape} | Features: {processed_dfs[name].shape[1]}')

print('\n📊 Feature List:')
feature_cols = [c for c in processed_dfs['ADANIGREEN'].columns 
                if c not in ['Date','Target']]
print(feature_cols)

---
# 🏆 REVIEW 1: Advanced Algorithms on Real Dataset
### Algorithms: Random Forest | XGBoost | SVR | LSTM

In [ ]:
# === Use ADANIGREEN as primary stock for Review 1 ===
PRIMARY = 'ADANIGREEN'
df_primary = processed_dfs[PRIMARY].copy()

FEATURES = [c for c in df_primary.columns if c not in ['Date', 'Target', 'Close']]
TARGET = 'Target'

X = df_primary[FEATURES].values
y = df_primary[TARGET].values

# Scale features
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1,1)).flatten()

split = int(0.8 * len(X_scaled))
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y_scaled[:split], y_scaled[split:]

y_test_actual = scaler_y.inverse_transform(y_test.reshape(-1,1)).flatten()

print(f'✅ Train: {X_train.shape} | Test: {X_test.shape}')
print(f'📊 Target range: {y.min():.2f} to {y.max():.2f}')

In [ ]:
# ===== ALGORITHM 1: Random Forest =====
print('🌲 Training Random Forest...')

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_pred_scaled = rf_model.predict(X_test)
rf_pred = scaler_y.inverse_transform(rf_pred_scaled.reshape(-1,1)).flatten()

rf_mae  = mean_absolute_error(y_test_actual, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test_actual, rf_pred))
rf_r2   = r2_score(y_test_actual, rf_pred)

print(f'\n🌲 Random Forest Results:')
print(f'   MAE  : {rf_mae:.4f}')
print(f'   RMSE : {rf_rmse:.4f}')
print(f'   R²   : {rf_r2:.4f}')

In [ ]:
# ===== ALGORITHM 2: XGBoost =====
print('⚡ Training XGBoost...')

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              verbose=False)

xgb_pred_scaled = xgb_model.predict(X_test)
xgb_pred = scaler_y.inverse_transform(xgb_pred_scaled.reshape(-1,1)).flatten()

xgb_mae  = mean_absolute_error(y_test_actual, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test_actual, xgb_pred))
xgb_r2   = r2_score(y_test_actual, xgb_pred)

print(f'\n⚡ XGBoost Results:')
print(f'   MAE  : {xgb_mae:.4f}')
print(f'   RMSE : {xgb_rmse:.4f}')
print(f'   R²   : {xgb_r2:.4f}')

In [ ]:
# ===== ALGORITHM 3: SVR =====
print('🔵 Training SVR...')

svr_model = SVR(kernel='rbf', C=100, gamma=0.01, epsilon=0.001)
svr_model.fit(X_train, y_train)

svr_pred_scaled = svr_model.predict(X_test)
svr_pred = scaler_y.inverse_transform(svr_pred_scaled.reshape(-1,1)).flatten()

svr_mae  = mean_absolute_error(y_test_actual, svr_pred)
svr_rmse = np.sqrt(mean_squared_error(y_test_actual, svr_pred))
svr_r2   = r2_score(y_test_actual, svr_pred)

print(f'\n🔵 SVR Results:')
print(f'   MAE  : {svr_mae:.4f}')
print(f'   RMSE : {svr_rmse:.4f}')
print(f'   R²   : {svr_r2:.4f}')

In [ ]:
# ===== ALGORITHM 4: LSTM (Deep Learning) =====
print('🧠 Training LSTM...')

LOOKBACK = 30

def create_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_scaled, LOOKBACK)

split_seq = int(0.8 * len(X_seq))
X_train_seq, X_test_seq = X_seq[:split_seq], X_seq[split_seq:]
y_train_seq, y_test_seq = y_seq[:split_seq], y_seq[split_seq:]

# Build LSTM Model
lstm_model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(LOOKBACK, X_scaled.shape[1])),
    Dropout(0.2),
    BatchNormalization(),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.summary()

early_stop = EarlyStopping(patience=10, restore_best_weights=True)

history = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

lstm_pred_scaled = lstm_model.predict(X_test_seq).flatten()
lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled.reshape(-1,1)).flatten()
y_test_lstm_actual = scaler_y.inverse_transform(y_test_seq.reshape(-1,1)).flatten()

lstm_mae  = mean_absolute_error(y_test_lstm_actual, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(y_test_lstm_actual, lstm_pred))
lstm_r2   = r2_score(y_test_lstm_actual, lstm_pred)

print(f'\n🧠 LSTM Results:')
print(f'   MAE  : {lstm_mae:.4f}')
print(f'   RMSE : {lstm_rmse:.4f}')
print(f'   R²   : {lstm_r2:.4f}')

In [ ]:
# ===== REVIEW 1: Results Comparison =====
results_real = pd.DataFrame({
    'Algorithm': ['Random Forest', 'XGBoost', 'SVR', 'LSTM'],
    'MAE':  [rf_mae, xgb_mae, svr_mae, lstm_mae],
    'RMSE': [rf_rmse, xgb_rmse, svr_rmse, lstm_rmse],
    'R²':   [rf_r2, xgb_r2, svr_r2, lstm_r2],
    'Dataset': ['Real'] * 4
})
print('\n🏆 REVIEW 1 — Algorithm Comparison on REAL Dataset:')
print(results_real.to_string(index=False))

In [ ]:
# Plot Predictions vs Actual
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

preds  = [rf_pred, xgb_pred, svr_pred, lstm_pred]
actuals = [y_test_actual, y_test_actual, y_test_actual, y_test_lstm_actual]
algo_names = ['Random Forest', 'XGBoost', 'SVR', 'LSTM']
colors_p = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for i, (name, pred, actual) in enumerate(zip(algo_names, preds, actuals)):
    axes[i].plot(actual, label='Actual', color='black', linewidth=1.5, alpha=0.7)
    axes[i].plot(pred,   label=f'{name} Predicted', color=colors_p[i], linewidth=1.5, linestyle='--')
    axes[i].set_title(f'{name} — Actual vs Predicted', fontsize=12, fontweight='bold')
    axes[i].legend()
    axes[i].set_xlabel('Time Steps')
    axes[i].set_ylabel('Stock Price (INR)')
    axes[i].grid(True, alpha=0.3)

plt.suptitle(f'📈 {PRIMARY} — All Algorithms: Actual vs Predicted', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('review1_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 🤖 REVIEW 2: GAN — Generate Synthetic Stock Dataset
### TimeGAN-style GAN for Time Series Synthesis

In [ ]:
# ===== GAN ARCHITECTURE =====
print('🤖 Building GAN for Synthetic Stock Data Generation...')

# Use Close prices + Volume for GAN training
df_gan = df_primary[['Open','High','Low','Close','Volume']].values
gan_scaler = MinMaxScaler()
df_gan_scaled = gan_scaler.fit_transform(df_gan)

SEQ_LEN   = 20
NOISE_DIM = 50
INPUT_DIM = df_gan_scaled.shape[1]  # 5 features

def make_sequences_gan(data, seq_len):
    seqs = []
    for i in range(len(data) - seq_len):
        seqs.append(data[i:i+seq_len])
    return np.array(seqs, dtype=np.float32)

real_seqs = make_sequences_gan(df_gan_scaled, SEQ_LEN)
real_seqs_torch = torch.tensor(real_seqs)
print(f'✅ Real sequences shape: {real_seqs.shape}')

# ---- Generator ----
class Generator(nn.Module):
    def __init__(self, noise_dim, input_dim, seq_len):
        super(Generator, self).__init__()
        self.seq_len   = seq_len
        self.input_dim = input_dim
        self.lstm = nn.LSTM(noise_dim, 128, num_layers=2, batch_first=True)
        self.fc   = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid()
        )
    def forward(self, z):
        out, _ = self.lstm(z)
        return self.fc(out)

# ---- Discriminator ----
class Discriminator(nn.Module):
    def __init__(self, input_dim, seq_len):
        super(Discriminator, self).__init__()
        self.lstm = nn.LSTM(input_dim, 128, num_layers=2, batch_first=True)
        self.fc   = nn.Sequential(
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator     = Generator(NOISE_DIM, INPUT_DIM, SEQ_LEN).to(device)
discriminator = Discriminator(INPUT_DIM, SEQ_LEN).to(device)

g_optimizer = torch.optim.Adam(generator.parameters(),     lr=0.0002, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))
criterion   = nn.BCELoss()

print(f'✅ GAN built | Device: {device}')
print(f'Generator parameters:     {sum(p.numel() for p in generator.parameters()):,}')
print(f'Discriminator parameters: {sum(p.numel() for p in discriminator.parameters()):,}')

In [ ]:
# ===== GAN TRAINING =====
EPOCHS     = 200
BATCH_SIZE = 32
g_losses, d_losses = [], []

print(f'🏋️ Training GAN for {EPOCHS} epochs...')

for epoch in range(EPOCHS):
    # Sample real batch
    idx        = np.random.randint(0, len(real_seqs), BATCH_SIZE)
    real_batch = real_seqs_torch[idx].to(device)
    
    real_labels = torch.ones(BATCH_SIZE, 1).to(device)
    fake_labels = torch.zeros(BATCH_SIZE, 1).to(device)
    
    # ----- Train Discriminator -----
    d_optimizer.zero_grad()
    d_real_loss = criterion(discriminator(real_batch), real_labels)
    
    noise      = torch.randn(BATCH_SIZE, SEQ_LEN, NOISE_DIM).to(device)
    fake_batch = generator(noise).detach()
    d_fake_loss = criterion(discriminator(fake_batch), fake_labels)
    
    d_loss = (d_real_loss + d_fake_loss) / 2
    d_loss.backward()
    d_optimizer.step()
    
    # ----- Train Generator -----
    g_optimizer.zero_grad()
    noise      = torch.randn(BATCH_SIZE, SEQ_LEN, NOISE_DIM).to(device)
    fake_batch = generator(noise)
    g_loss     = criterion(discriminator(fake_batch), real_labels)
    g_loss.backward()
    g_optimizer.step()
    
    g_losses.append(g_loss.item())
    d_losses.append(d_loss.item())
    
    if (epoch+1) % 50 == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}] | G Loss: {g_loss.item():.4f} | D Loss: {d_loss.item():.4f}')

print('\n✅ GAN Training Complete!')

# Plot GAN loss
plt.figure(figsize=(10, 4))
plt.plot(g_losses, label='Generator Loss',     color='blue')
plt.plot(d_losses, label='Discriminator Loss', color='red')
plt.title('GAN Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('gan_loss.png', dpi=150)
plt.show()

In [ ]:
# ===== GENERATE SYNTHETIC DATASET =====
print('🎲 Generating Synthetic Stock Data...')

NUM_SAMPLES = len(real_seqs)  # Same size as real data

generator.eval()
with torch.no_grad():
    noise   = torch.randn(NUM_SAMPLES, SEQ_LEN, NOISE_DIM).to(device)
    fake    = generator(noise).cpu().numpy()

# Take last timestep of each sequence as a row
synthetic_scaled = fake[:, -1, :]  # shape: (N, 5)
synthetic_data   = gan_scaler.inverse_transform(synthetic_scaled)

synthetic_df = pd.DataFrame(synthetic_data, columns=['Open','High','Low','Close','Volume'])

print(f'✅ Synthetic dataset shape: {synthetic_df.shape}')
print(synthetic_df.head())

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_primary['Close'], bins=50, alpha=0.7, label='Real',      color='#2196F3')
axes[0].hist(synthetic_df['Close'],bins=50, alpha=0.7, label='Synthetic', color='#FF5722')
axes[0].set_title('Close Price Distribution: Real vs Synthetic', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(df_primary['Volume'], bins=50, alpha=0.7, label='Real',      color='#2196F3')
axes[1].hist(synthetic_df['Volume'],bins=50, alpha=0.7, label='Synthetic', color='#FF5722')
axes[1].set_title('Volume Distribution: Real vs Synthetic', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('🤖 GAN: Real vs Synthetic Data Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gan_distributions.png', dpi=150)
plt.show()

In [ ]:
# ===== Run Algorithms on GAN-Generated Dataset =====
print('🔬 Running Algorithms on GAN-Generated Synthetic Dataset...')

# Add technical indicators to synthetic data
# (synthetic data doesn't have Date, so we build manually)
syn = synthetic_df.copy()
syn['MA_7']  = syn['Close'].rolling(7, min_periods=1).mean()
syn['MA_21'] = syn['Close'].rolling(21, min_periods=1).mean()
syn['RSI']   = 50  # simplified placeholder for synthetic
syn['MACD']  = syn['Close'].ewm(span=12).mean() - syn['Close'].ewm(span=26).mean()
syn['Daily_Return'] = syn['Close'].pct_change().fillna(0)
syn['Volatility']   = syn['Daily_Return'].rolling(14, min_periods=1).std().fillna(0)
syn['Momentum']     = syn['Close'] - syn['Close'].shift(10).fillna(method='bfill')
syn['Volume_Ratio'] = syn['Volume'] / (syn['Volume'].rolling(10, min_periods=1).mean() + 1e-8)
syn['Target'] = syn['Close'].shift(-1)
syn.dropna(inplace=True)

syn_feat = [c for c in syn.columns if c not in ['Target']]
Xs = syn[syn_feat].values
ys = syn['Target'].values

sX_scaler = MinMaxScaler()
sy_scaler = MinMaxScaler()
Xs_scaled = sX_scaler.fit_transform(Xs)
ys_scaled = sy_scaler.fit_transform(ys.reshape(-1,1)).flatten()

ssplit = int(0.8 * len(Xs_scaled))
sX_train, sX_test = Xs_scaled[:ssplit], Xs_scaled[ssplit:]
sy_train, sy_test = ys_scaled[:ssplit], ys_scaled[ssplit:]
sy_actual = sy_scaler.inverse_transform(sy_test.reshape(-1,1)).flatten()

def evaluate(model, X_tr, y_tr, X_te, y_te_actual, scaler):
    model.fit(X_tr, y_tr)
    pred_s = model.predict(X_te)
    pred   = scaler.inverse_transform(pred_s.reshape(-1,1)).flatten()
    return (
        mean_absolute_error(y_te_actual, pred),
        np.sqrt(mean_squared_error(y_te_actual, pred)),
        r2_score(y_te_actual, pred)
    )

s_rf_mae, s_rf_rmse, s_rf_r2 = evaluate(
    RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    sX_train, sy_train, sX_test, sy_actual, sy_scaler)

s_xgb_mae, s_xgb_rmse, s_xgb_r2 = evaluate(
    xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, verbosity=0),
    sX_train, sy_train, sX_test, sy_actual, sy_scaler)

s_svr_mae, s_svr_rmse, s_svr_r2 = evaluate(
    SVR(kernel='rbf', C=100, gamma=0.01),
    sX_train, sy_train, sX_test, sy_actual, sy_scaler)

results_synthetic = pd.DataFrame({
    'Algorithm': ['Random Forest', 'XGBoost', 'SVR'],
    'MAE':  [s_rf_mae,  s_xgb_mae,  s_svr_mae],
    'RMSE': [s_rf_rmse, s_xgb_rmse, s_svr_rmse],
    'R²':   [s_rf_r2,   s_xgb_r2,   s_svr_r2],
    'Dataset': ['GAN-Synthetic'] * 3
})
print('\n🤖 GAN-Synthetic Dataset Results:')
print(results_synthetic.to_string(index=False))

In [ ]:
# ===== REVIEW 2: COMPARISON SLIDE (Real vs GAN Synthetic) =====
compare_df = pd.concat([
    results_real[results_real['Algorithm'] != 'LSTM'],
    results_synthetic
], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['MAE', 'RMSE', 'R²']
palette = {'Real': '#2196F3', 'GAN-Synthetic': '#FF5722'}

for i, metric in enumerate(metrics):
    real_vals = compare_df[compare_df['Dataset']=='Real'][metric].values
    synt_vals = compare_df[compare_df['Dataset']=='GAN-Synthetic'][metric].values
    algos     = compare_df[compare_df['Dataset']=='Real']['Algorithm'].values
    
    x = np.arange(len(algos))
    w = 0.35
    axes[i].bar(x - w/2, real_vals, w, label='Real Dataset',      color='#2196F3', alpha=0.85)
    axes[i].bar(x + w/2, synt_vals, w, label='GAN-Synthetic',     color='#FF5722', alpha=0.85)
    axes[i].set_title(f'{metric} Comparison', fontweight='bold', fontsize=13)
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(algos, rotation=15)
    axes[i].legend()
    axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('📊 REVIEW 2: Real Dataset vs GAN-Synthetic Dataset — Algorithm Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('review2_comparison_slide.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison slide saved as review2_comparison_slide.png')

---
# 📰 REVIEW 3: Text Mining — Sentiment Analysis
### Simulated News Headlines → Sentiment Score → Correlation with Stock

In [ ]:
# ===== TEXT MINING PIPELINE =====
print('📰 Text Mining Module...')

# Sample news headlines (in real project: scrape from moneycontrol/economictimes)
sample_headlines = {
    'ADANIGREEN': [
        "Adani Green Energy reports record renewable capacity addition",
        "Adani Green faces regulatory scrutiny over solar project delays",
        "Adani Green stock surges on strong quarterly earnings beat",
        "Adani Green wins 1500 MW solar project in Rajasthan",
        "Market selloff drags Adani Green to multi-month low",
        "Adani Green partners with global fund for green hydrogen",
        "Adani Group faces headwinds from Hindenburg report concerns",
        "Adani Green Energy secures massive international investment"
    ],
    'INOXGREEN': [
        "INOX Green Energy posts strong revenue growth in Q2",
        "INOX Green expands wind energy portfolio in Madhya Pradesh",
        "INOX Green signs PPA for 400 MW wind farm",
        "INOX Green shares fall on profit booking pressure",
        "INOX Green raises funds for capacity expansion"
    ],
    'JSWENERGY': [
        "JSW Energy beats estimates with strong Q3 performance",
        "JSW Energy accelerates renewable transition plan",
        "JSW Energy acquires hydro power assets in Himachal Pradesh",
        "JSW Energy faces cost pressure from fuel price spike",
        "JSW Energy stock rises on positive analyst upgrade"
    ],
    'SUZLON': [
        "Suzlon Energy wins 500 MW wind order from NTPC",
        "Suzlon clears debt, sets stage for massive growth",
        "Suzlon Energy reports strong order book for FY25",
        "Suzlon shares consolidate after sharp rally",
        "Suzlon Energy targets 2GW capacity addition next year",
        "Suzlon Energy faces supply chain delays in turbine delivery"
    ]
}

# Text Preprocessing
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return ' '.join(tokens)

def get_sentiment(text):
    analysis = TextBlob(text)
    polarity = analysis.sentiment.polarity
    if   polarity > 0.1:  label = 'Positive'
    elif polarity < -0.1: label = 'Negative'
    else:                  label = 'Neutral'
    return polarity, label

all_sentiment_rows = []

for stock, headlines in sample_headlines.items():
    for h in headlines:
        cleaned = preprocess_text(h)
        polarity, label = get_sentiment(h)
        all_sentiment_rows.append({
            'Stock': stock,
            'Headline': h,
            'Cleaned': cleaned,
            'Polarity': polarity,
            'Sentiment': label
        })

sentiment_df = pd.DataFrame(all_sentiment_rows)
print(sentiment_df.to_string(index=False))

In [ ]:
# ===== Sentiment Visualization =====
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sentiment distribution by stock
sentiment_summary = sentiment_df.groupby(['Stock','Sentiment']).size().unstack(fill_value=0)
sentiment_summary.plot(kind='bar', ax=axes[0], color=['#4CAF50','#9E9E9E','#F44336'], alpha=0.85)
axes[0].set_title('Sentiment Distribution per Stock', fontweight='bold')
axes[0].set_xlabel('Stock')
axes[0].set_ylabel('Count')
axes[0].legend(title='Sentiment')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(True, alpha=0.3, axis='y')

# Average polarity
avg_polarity = sentiment_df.groupby('Stock')['Polarity'].mean()
colors_bar = ['#4CAF50' if v > 0 else '#F44336' for v in avg_polarity]
avg_polarity.plot(kind='bar', ax=axes[1], color=colors_bar, alpha=0.85)
axes[1].set_title('Average Sentiment Polarity per Stock', fontweight='bold')
axes[1].set_xlabel('Stock')
axes[1].set_ylabel('Polarity Score')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('📰 REVIEW 3: Text Mining — Stock News Sentiment Analysis',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('review3_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Sentiment analysis saved!')

In [ ]:
# Correlation: Sentiment Polarity vs Next-day Return
# (Simulated mapping for demonstration)
random.seed(42)
np.random.seed(42)

sentiment_df['Next_Day_Return'] = sentiment_df['Polarity'] * 0.8 + np.random.normal(0, 0.01, len(sentiment_df))

plt.figure(figsize=(8, 5))
colors_map = {'Positive': '#4CAF50', 'Neutral': '#9E9E9E', 'Negative': '#F44336'}
for label, group in sentiment_df.groupby('Sentiment'):
    plt.scatter(group['Polarity'], group['Next_Day_Return'],
                label=label, color=colors_map[label], alpha=0.8, s=80)

m, b = np.polyfit(sentiment_df['Polarity'], sentiment_df['Next_Day_Return'], 1)
x_line = np.linspace(sentiment_df['Polarity'].min(), sentiment_df['Polarity'].max(), 100)
plt.plot(x_line, m*x_line + b, 'k--', linewidth=1.5, label='Trend Line')
plt.title('Sentiment Polarity vs Next-Day Stock Return', fontweight='bold')
plt.xlabel('Sentiment Polarity Score')
plt.ylabel('Next-Day Return')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('sentiment_correlation.png', dpi=150)
plt.show()

corr = sentiment_df['Polarity'].corr(sentiment_df['Next_Day_Return'])
print(f'📊 Pearson Correlation (Sentiment ↔ Return): {corr:.4f}')

---
# 🤖 AI AGENT: Autonomous Stock Prediction & Recommendation
### The Agent uses best model + sentiment to make decisions

In [ ]:
# ===== TRAIN BEST MODEL FOR EACH STOCK =====
print('🏗️ Training Best Model (XGBoost) for All 4 Stocks...')

stock_agents = {}

for stock_name, df in processed_dfs.items():
    feat_cols = [c for c in df.columns if c not in ['Date','Target']]
    X_s = df[feat_cols].values
    y_s = df['Target'].values
    
    sx = MinMaxScaler()
    sy = MinMaxScaler()
    X_s_sc = sx.fit_transform(X_s)
    y_s_sc = sy.fit_transform(y_s.reshape(-1,1)).flatten()
    
    spl = int(0.8 * len(X_s_sc))
    Xtr, Xte = X_s_sc[:spl], X_s_sc[spl:]
    ytr, yte = y_s_sc[:spl], y_s_sc[spl:]
    
    model_s = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05, verbosity=0)
    model_s.fit(Xtr, ytr)
    
    ypred_sc = model_s.predict(Xte)
    ypred    = sy.inverse_transform(ypred_sc.reshape(-1,1)).flatten()
    yact     = sy.inverse_transform(yte.reshape(-1,1)).flatten()
    
    r2 = r2_score(yact, ypred)
    
    stock_agents[stock_name] = {
        'model':       model_s,
        'scaler_X':    sx,
        'scaler_y':    sy,
        'feature_cols': feat_cols,
        'df':          df,
        'r2':          r2
    }
    print(f'  ✅ {stock_name}: R² = {r2:.4f}')

print('\n✅ All stock agents ready!')

In [ ]:
# ===== AI AGENT CORE FUNCTION =====

def get_latest_sentiment(stock_name):
    """Returns average sentiment polarity for a stock from news headlines."""
    headlines = sample_headlines.get(stock_name, [])
    if not headlines:
        return 0.0, 'Neutral'
    polarities = [get_sentiment(h)[0] for h in headlines]
    avg_pol = np.mean(polarities)
    if   avg_pol > 0.05:  label = 'Positive 📈'
    elif avg_pol < -0.05: label = 'Negative 📉'
    else:                  label = 'Neutral  ➡️'
    return round(avg_pol, 4), label


def generate_recommendation(predicted_price, current_price, sentiment_score):
    """AI Agent decision logic using price prediction + sentiment."""
    price_change_pct = (predicted_price - current_price) / (current_price + 1e-8) * 100
    
    # Decision matrix
    if   price_change_pct > 3  and sentiment_score > 0.1:  action = '🟢 STRONG BUY'
    elif price_change_pct > 1  and sentiment_score >= 0:   action = '🟩 BUY'
    elif price_change_pct < -3 and sentiment_score < -0.1: action = '🔴 STRONG SELL'
    elif price_change_pct < -1 and sentiment_score <= 0:   action = '🟥 SELL'
    else:                                                   action = '🟡 HOLD'
    
    confidence = min(abs(price_change_pct) / 10 * 100, 99)  # rough confidence score
    return action, round(price_change_pct, 2), round(confidence, 1)


def stock_prediction_agent(stock_name):
    """Main AI Agent: Predicts next-day price & gives recommendation."""
    agent_info = stock_agents[stock_name]
    df         = agent_info['df']
    model      = agent_info['model']
    sx         = agent_info['scaler_X']
    sy         = agent_info['scaler_y']
    feat_cols  = agent_info['feature_cols']
    
    # Use latest available row as input
    latest_row = df[feat_cols].iloc[-1:].values
    latest_scaled = sx.transform(latest_row)
    
    pred_scaled = model.predict(latest_scaled)
    predicted_price = sy.inverse_transform(pred_scaled.reshape(-1,1)).flatten()[0]
    current_price   = df['Close'].iloc[-1]
    
    sentiment_score, sentiment_label = get_latest_sentiment(stock_name)
    action, pct_change, confidence   = generate_recommendation(
        predicted_price, current_price, sentiment_score)
    
    return {
        'Stock':            stock_name,
        'Current_Price':    round(current_price, 2),
        'Predicted_Price':  round(predicted_price, 2),
        'Price_Change_%':   pct_change,
        'Sentiment_Score':  sentiment_score,
        'Sentiment_Label':  sentiment_label,
        'Recommendation':   action,
        'Confidence_%':     confidence,
        'Model_R²':         round(agent_info['r2'], 4)
    }

print('✅ AI Agent function defined!')

In [ ]:
# ===== RUN AI AGENT FOR ALL 4 STOCKS =====
print('\n' + '='*65)
print('  🤖 STOCK PREDICTION AI AGENT — GREEN ENERGY SECTOR')
print('='*65)

agent_results = []
for stock in ['ADANIGREEN', 'INOXGREEN', 'JSWENERGY', 'SUZLON']:
    result = stock_prediction_agent(stock)
    agent_results.append(result)
    
    print(f'\n📊 Stock         : {result["Stock"]}')
    print(f'   Current Price  : ₹{result["Current_Price"]}')
    print(f'   Predicted Next : ₹{result["Predicted_Price"]}')
    print(f'   Price Change   : {result["Price_Change_%"]}%')
    print(f'   Sentiment      : {result["Sentiment_Label"]} (score: {result["Sentiment_Score"]})')
    print(f'   Recommendation : {result["Recommendation"]}')
    print(f'   Confidence     : {result["Confidence_%"]}%')
    print(f'   Model R²       : {result["Model_R²"]}')
    print('-'*55)

print('\n✅ AI Agent Analysis Complete!')

In [ ]:
# ===== AI AGENT DASHBOARD =====
agent_df = pd.DataFrame(agent_results)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Plot 1: Current vs Predicted Price
x = np.arange(len(agent_df))
w = 0.35
axes[0,0].bar(x - w/2, agent_df['Current_Price'],   w, label='Current Price',   color='#2196F3', alpha=0.85)
axes[0,0].bar(x + w/2, agent_df['Predicted_Price'], w, label='Predicted Price', color='#4CAF50', alpha=0.85)
axes[0,0].set_title('Current vs Predicted Price', fontweight='bold')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels(agent_df['Stock'], rotation=10)
axes[0,0].set_ylabel('Price (₹ INR)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3, axis='y')

# Plot 2: Predicted Price Change %
colors_pct = ['#4CAF50' if v > 0 else '#F44336' for v in agent_df['Price_Change_%']]
axes[0,1].bar(agent_df['Stock'], agent_df['Price_Change_%'], color=colors_pct, alpha=0.85)
axes[0,1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[0,1].set_title('Predicted Price Change (%)', fontweight='bold')
axes[0,1].set_ylabel('Change %')
axes[0,1].tick_params(axis='x', rotation=10)
axes[0,1].grid(True, alpha=0.3, axis='y')

# Plot 3: Sentiment Scores
col_sent = ['#4CAF50' if v > 0 else '#F44336' for v in agent_df['Sentiment_Score']]
axes[1,0].bar(agent_df['Stock'], agent_df['Sentiment_Score'], color=col_sent, alpha=0.85)
axes[1,0].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1,0].set_title('News Sentiment Polarity Score', fontweight='bold')
axes[1,0].set_ylabel('Polarity')
axes[1,0].tick_params(axis='x', rotation=10)
axes[1,0].grid(True, alpha=0.3, axis='y')

# Plot 4: Model R² & Confidence
axes[1,1].bar(x - w/2, agent_df['Model_R²'] * 100, w, label='Model R² (%)',  color='#9C27B0', alpha=0.85)
axes[1,1].bar(x + w/2, agent_df['Confidence_%'],   w, label='Confidence (%)', color='#FF9800', alpha=0.85)
axes[1,1].set_title('Model R² vs Agent Confidence', fontweight='bold')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(agent_df['Stock'], rotation=10)
axes[1,1].set_ylabel('%')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.suptitle('🤖 AI AGENT DASHBOARD — Green Energy Stocks',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('ai_agent_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Dashboard saved as ai_agent_dashboard.png')

In [ ]:
# ===== INTERACTIVE AI AGENT (Type stock name to get prediction) =====

def run_agent_interactive():
    print('\n' + '='*60)
    print('  🤖 INTERACTIVE AI AGENT — Stock Prediction System')
    print('  Available: ADANIGREEN | INOXGREEN | JSWENERGY | SUZLON')
    print('='*60)
    
    stock_input = input('\nEnter Stock Name (or ALL): ').strip().upper()
    
    if stock_input == 'ALL':
        targets = list(stock_agents.keys())
    elif stock_input in stock_agents:
        targets = [stock_input]
    else:
        print(f'❌ Stock "{stock_input}" not found.')
        return
    
    for s in targets:
        r = stock_prediction_agent(s)
        print(f'\n{'─'*55}')
        print(f'📌 {r["Stock"]}')
        print(f'   Current Price  : ₹{r["Current_Price"]}')
        print(f'   Predicted Next : ₹{r["Predicted_Price"]}')
        print(f'   Change         : {r["Price_Change_%"]}%')
        print(f'   Sentiment      : {r["Sentiment_Label"]}')
        print(f'   Decision       : {r["Recommendation"]}')
        print(f'   Confidence     : {r["Confidence_%"]}%')

run_agent_interactive()

---
# 📋 Final Summary Report

In [ ]:
print('\n' + '='*65)
print('  📋 FDS PROJECT FINAL SUMMARY REPORT')
print('  Green Energy Stock Price Prediction — AI Agent System')
print('='*65)

print('\n📌 REVIEW 1 — Advanced Algorithms on Real Dataset:')
print(results_real[['Algorithm','MAE','RMSE','R²']].to_string(index=False))

print('\n📌 REVIEW 2 — GAN Synthetic vs Real Dataset Comparison:')
print(compare_df[['Algorithm','Dataset','MAE','RMSE','R²']].to_string(index=False))

print('\n📌 REVIEW 3 — Text Mining Sentiment Summary:')
sent_summary = sentiment_df.groupby('Stock').agg(
    Avg_Polarity=('Polarity','mean'),
    Positive_Headlines=('Sentiment', lambda x: (x=='Positive').sum()),
    Negative_Headlines=('Sentiment', lambda x: (x=='Negative').sum())
).reset_index()
print(sent_summary.to_string(index=False))

print('\n📌 AI AGENT — Final Recommendations:')
print(agent_df[['Stock','Current_Price','Predicted_Price','Recommendation','Confidence_%']].to_string(index=False))

print('\n✅ All Reviews Complete. AI Agent Ready.')
print('📁 Output Files: stock_trends.png | review1_predictions.png |')
print('                 gan_loss.png | gan_distributions.png |')
print('                 review2_comparison_slide.png | review3_sentiment.png |')
print('                 ai_agent_dashboard.png')
print('='*65)